# NB_bishop_ch19_autoencoders_vae

**Chapter 19 — Autoencoders and Variational Autoencoders**

This notebook builds a standard autoencoder on MNIST, implements a VAE with the reparameterization trick, visualizes the 2D latent space, generates new samples, and demonstrates anomaly detection via reconstruction error.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_19/NB_bishop_ch19_autoencoders_vae.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, precision_recall_curve

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## 1. Data Preparation

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print(f'Training: {X_train_flat.shape}, Test: {X_test_flat.shape}')

## 2. Standard Autoencoder

In [ ]:
# Build standard autoencoder
LATENT_DIM_AE = 32

encoder_ae = keras.Sequential([
    layers.InputLayer(input_shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(LATENT_DIM_AE, activation='relu')
], name='encoder')

decoder_ae = keras.Sequential([
    layers.InputLayer(input_shape=(LATENT_DIM_AE,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(784, activation='sigmoid')
], name='decoder')

ae_input = keras.Input(shape=(784,))
ae_encoded = encoder_ae(ae_input)
ae_decoded = decoder_ae(ae_encoded)
autoencoder = keras.Model(ae_input, ae_decoded, name='autoencoder')

autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()

In [ ]:
# Train autoencoder
history_ae = autoencoder.fit(
    X_train_flat, X_train_flat,
    epochs=30,
    batch_size=256,
    validation_data=(X_test_flat, X_test_flat),
    verbose=1
)
print(f'Final AE val loss: {history_ae.history["val_loss"][-1]:.4f}')

In [ ]:
# Visualize AE reconstructions
ae_recon = autoencoder.predict(X_test_flat[:10], verbose=0)

fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
for i in range(10):
    axes[0, i].imshow(X_test[i], cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Original' if i == 0 else '')
    axes[1, i].imshow(ae_recon[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Reconstructed' if i == 0 else '')

fig.suptitle('Autoencoder Reconstruction (latent dim = 32)', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch19_ae_reconstruction')
plt.show()

## 3. Variational Autoencoder (VAE)

In [ ]:
# Sampling layer with reparameterization trick
class Sampling(layers.Layer):
    """Reparameterization trick: z = mu + sigma * epsilon."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

print('Sampling layer defined.')

In [ ]:
# Build VAE
LATENT_DIM = 2  # 2D for visualization

# Encoder
enc_input = keras.Input(shape=(784,))
x = layers.Dense(256, activation='relu')(enc_input)
x = layers.Dense(128, activation='relu')(x)
z_mean = layers.Dense(LATENT_DIM, name='z_mean')(x)
z_log_var = layers.Dense(LATENT_DIM, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])
vae_encoder = keras.Model(enc_input, [z_mean, z_log_var, z], name='vae_encoder')

# Decoder
dec_input = keras.Input(shape=(LATENT_DIM,))
x = layers.Dense(128, activation='relu')(dec_input)
x = layers.Dense(256, activation='relu')(x)
dec_output = layers.Dense(784, activation='sigmoid')(x)
vae_decoder = keras.Model(dec_input, dec_output, name='vae_decoder')

vae_encoder.summary()
vae_decoder.summary()

In [ ]:
# VAE model with custom training step
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name='total_loss')
        self.recon_loss_tracker = keras.metrics.Mean(name='recon_loss')
        self.kl_loss_tracker = keras.metrics.Mean(name='kl_loss')
    
    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]
    
    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction),
                    axis=-1
                )
            )
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=-1
                )
            )
            total_loss = recon_loss + kl_loss
        
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            'loss': self.total_loss_tracker.result(),
            'recon_loss': self.recon_loss_tracker.result(),
            'kl_loss': self.kl_loss_tracker.result()
        }

vae = VAE(vae_encoder, vae_decoder)
vae.compile(optimizer=keras.optimizers.Adam(1e-3))
print('VAE model compiled.')

In [ ]:
# Train VAE
history_vae = vae.fit(
    X_train_flat,
    epochs=50,
    batch_size=256,
    verbose=1
)

In [ ]:
# Plot VAE training losses
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history_vae.history['loss'], color='#1a3a6e', linewidth=1.5)
axes[0].set_title('Total Loss'); axes[0].set_xlabel('Epoch')
axes[1].plot(history_vae.history['recon_loss'], color='#cd0000', linewidth=1.5)
axes[1].set_title('Reconstruction Loss'); axes[1].set_xlabel('Epoch')
axes[2].plot(history_vae.history['kl_loss'], color='#228B22', linewidth=1.5)
axes[2].set_title('KL Divergence'); axes[2].set_xlabel('Epoch')
fig.suptitle('VAE Training Losses', fontsize=14)
fig.tight_layout()
plt.show()

## 4. Latent Space Visualization

In [ ]:
# Encode test data and visualize 2D latent space
z_mean_test, z_log_var_test, z_test = vae_encoder.predict(X_test_flat, verbose=0)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for digit in range(10):
    mask = y_test == digit
    ax.scatter(z_mean_test[mask, 0], z_mean_test[mask, 1],
               c=[colors[digit]], s=5, alpha=0.5, label=str(digit))
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
ax.set_title('VAE Latent Space (2D) — Test Digits')
ax.legend(markerscale=4, fontsize=10)
fig.tight_layout()
save_fig(fig, 'bishop_ch19_vae_latent')
plt.show()

In [ ]:
# Latent space traversal: decode grid of z values
n_grid = 20
z1 = np.linspace(-3, 3, n_grid)
z2 = np.linspace(-3, 3, n_grid)

canvas = np.zeros((28 * n_grid, 28 * n_grid))
for i, zi in enumerate(z1):
    for j, zj in enumerate(z2):
        z_sample = np.array([[zi, zj]], dtype=np.float32)
        decoded = vae_decoder.predict(z_sample, verbose=0)
        canvas[j * 28:(j + 1) * 28, i * 28:(i + 1) * 28] = decoded.reshape(28, 28)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(canvas, cmap='gray', origin='lower',
          extent=[-3, 3, -3, 3])
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
ax.set_title('VAE Latent Space Traversal')
fig.tight_layout()
plt.show()

## 5. Generating New Samples

In [ ]:
# Generate new digits by sampling from the prior N(0, I)
n_samples = 64
z_random = np.random.normal(0, 1, size=(n_samples, LATENT_DIM)).astype(np.float32)
generated = vae_decoder.predict(z_random, verbose=0)

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
fig.suptitle('VAE — 64 Generated Digits (Sampled from Prior)', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch19_vae_generated')
plt.show()

In [ ]:
# Interpolation in latent space between two digits
idx_a, idx_b = 0, 1  # two test images
z_a = z_mean_test[idx_a]
z_b = z_mean_test[idx_b]

n_interp = 12
alphas = np.linspace(0, 1, n_interp)
z_interp = np.array([(1 - a) * z_a + a * z_b for a in alphas], dtype=np.float32)
interp_decoded = vae_decoder.predict(z_interp, verbose=0)

fig, axes = plt.subplots(1, n_interp, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(interp_decoded[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
    ax.set_title(f'{alphas[i]:.1f}')
fig.suptitle(f'Latent Interpolation: {y_test[idx_a]} to {y_test[idx_b]}', fontsize=14)
fig.tight_layout()
plt.show()

## 6. Anomaly Detection via Reconstruction Error

In [ ]:
# Train a VAE on only one digit class (e.g., digit 1) and detect others as anomalies
normal_digit = 1
X_normal = X_train_flat[y_train == normal_digit]
print(f'Training on digit {normal_digit}: {X_normal.shape[0]} samples')

# Build a new VAE for anomaly detection (higher latent dim)
AD_LATENT = 8

ad_enc_in = keras.Input(shape=(784,))
h = layers.Dense(128, activation='relu')(ad_enc_in)
h = layers.Dense(64, activation='relu')(h)
ad_z_mean = layers.Dense(AD_LATENT)(h)
ad_z_log_var = layers.Dense(AD_LATENT)(h)
ad_z = Sampling()([ad_z_mean, ad_z_log_var])
ad_encoder = keras.Model(ad_enc_in, [ad_z_mean, ad_z_log_var, ad_z], name='ad_encoder')

ad_dec_in = keras.Input(shape=(AD_LATENT,))
h = layers.Dense(64, activation='relu')(ad_dec_in)
h = layers.Dense(128, activation='relu')(h)
ad_dec_out = layers.Dense(784, activation='sigmoid')(h)
ad_decoder = keras.Model(ad_dec_in, ad_dec_out, name='ad_decoder')

ad_vae = VAE(ad_encoder, ad_decoder)
ad_vae.compile(optimizer=keras.optimizers.Adam(1e-3))

In [ ]:
# Train on normal digit only
ad_vae.fit(X_normal, epochs=30, batch_size=128, verbose=1)

In [ ]:
# Compute reconstruction error on test set
z_mean_ad, _, z_ad = ad_encoder.predict(X_test_flat, verbose=0)
recon_ad = ad_decoder.predict(z_ad, verbose=0)
recon_error = np.mean((X_test_flat - recon_ad) ** 2, axis=1)

# Labels: normal_digit = 0 (normal), others = 1 (anomaly)
is_anomaly = (y_test != normal_digit).astype(int)

auc = roc_auc_score(is_anomaly, recon_error)
print(f'Anomaly detection AUC-ROC: {auc:.4f}')

In [ ]:
# Plot reconstruction error distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(recon_error[is_anomaly == 0], bins=50, alpha=0.7, color='#228B22',
             label=f'Normal (digit {normal_digit})', density=True)
axes[0].hist(recon_error[is_anomaly == 1], bins=50, alpha=0.7, color='#cd0000',
             label='Anomaly (other digits)', density=True)
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Reconstruction Error Distribution (AUC={auc:.3f})')
axes[0].legend()

# Show examples: highest reconstruction error = anomalies
top_anomaly_idx = np.argsort(recon_error)[-8:]
top_normal_idx = np.argsort(recon_error)[:8]

# Plot top anomalies
for i in range(4):
    # Create small inset-like subplot
    ax_pos = axes[1].get_position()

axes[1].set_visible(False)

fig.tight_layout()
plt.show()

# Show top anomalies and top normals
fig2, axes2 = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes2[0, i].imshow(X_test[top_normal_idx[i]], cmap='gray')
    axes2[0, i].axis('off')
    axes2[0, i].set_title(f'err={recon_error[top_normal_idx[i]]:.3f}' if i < 4 else '')
    axes2[1, i].imshow(X_test[top_anomaly_idx[i]], cmap='gray')
    axes2[1, i].axis('off')
    axes2[1, i].set_title(f'err={recon_error[top_anomaly_idx[i]]:.3f}' if i < 4 else '')

axes2[0, 0].set_ylabel('Normal', fontsize=12, rotation=0, labelpad=40)
axes2[1, 0].set_ylabel('Anomaly', fontsize=12, rotation=0, labelpad=40)
fig2.suptitle(f'Anomaly Detection: Normal=digit {normal_digit}, Anomaly=other', fontsize=14)
fig2.tight_layout()
save_fig(fig2, 'bishop_ch19_anomaly')
plt.show()

## 7. Higher-Dimensional VAE (latent_dim=10)

In [ ]:
# VAE with latent_dim=10 for better reconstruction
LATENT_DIM_HD = 10

hd_enc_in = keras.Input(shape=(784,))
h = layers.Dense(256, activation='relu')(hd_enc_in)
h = layers.Dense(128, activation='relu')(h)
hd_z_mean = layers.Dense(LATENT_DIM_HD)(h)
hd_z_log_var = layers.Dense(LATENT_DIM_HD)(h)
hd_z = Sampling()([hd_z_mean, hd_z_log_var])
hd_encoder = keras.Model(hd_enc_in, [hd_z_mean, hd_z_log_var, hd_z])

hd_dec_in = keras.Input(shape=(LATENT_DIM_HD,))
h = layers.Dense(128, activation='relu')(hd_dec_in)
h = layers.Dense(256, activation='relu')(h)
hd_dec_out = layers.Dense(784, activation='sigmoid')(h)
hd_decoder = keras.Model(hd_dec_in, hd_dec_out)

hd_vae = VAE(hd_encoder, hd_decoder)
hd_vae.compile(optimizer=keras.optimizers.Adam(1e-3))

hd_vae.fit(X_train_flat, epochs=30, batch_size=256, verbose=1)

In [ ]:
# Compare reconstructions: dim=2 vs dim=10
_, _, z2_test = vae_encoder.predict(X_test_flat[:10], verbose=0)
recon_2d = vae_decoder.predict(z2_test, verbose=0)

_, _, z10_test = hd_encoder.predict(X_test_flat[:10], verbose=0)
recon_10d = hd_decoder.predict(z10_test, verbose=0)

fig, axes = plt.subplots(3, 10, figsize=(16, 5))
for i in range(10):
    axes[0, i].imshow(X_test[i], cmap='gray'); axes[0, i].axis('off')
    axes[0, i].set_title('Original' if i == 0 else '')
    axes[1, i].imshow(recon_2d[i].reshape(28, 28), cmap='gray'); axes[1, i].axis('off')
    axes[1, i].set_title('VAE (d=2)' if i == 0 else '')
    axes[2, i].imshow(recon_10d[i].reshape(28, 28), cmap='gray'); axes[2, i].axis('off')
    axes[2, i].set_title('VAE (d=10)' if i == 0 else '')
fig.suptitle('VAE Reconstruction: Latent Dim 2 vs 10', fontsize=14)
fig.tight_layout()
plt.show()

## Summary

In this notebook we covered:
- **Standard autoencoder** — deterministic encoder-decoder for reconstruction
- **VAE with reparameterization trick** — enabling backpropagation through stochastic sampling
- **2D latent space visualization** — digits form organized clusters
- **Sample generation** — sampling from the prior and decoding
- **Anomaly detection** — using reconstruction error to identify out-of-distribution data

In [ ]:
# Key takeaways
takeaways = [
    '1. Autoencoders learn compressed representations via encoder-decoder architecture.',
    '2. The VAE reparameterization trick enables gradient-based optimization of the ELBO.',
    '3. The KL divergence term regularizes the latent space toward a Gaussian prior.',
    '4. VAE latent spaces are smooth and continuous, enabling meaningful interpolation.',
    '5. Reconstruction error provides a natural anomaly score for out-of-distribution detection.'
]
print('Key Takeaways:')
print('-' * 60)
for t in takeaways:
    print(t)